# RAMTA Multimodal Feasibility Pilot (Text + Audio + Visual + RAF)
**Thesis:** Analysis of the Impact of Political Multimodal Discourse on Financial Markets — Melike Nazlı Gürbüz (s34853)

This notebook implements and runs the **full multimodal RAMTA architecture end-to-end**
on N=5 real political events (FOMC press conferences, 2015-2024), each with real
text, audio, and video sourced from federalreserve.gov.

Because N=5 is too small for a statistically powered accuracy evaluation, this
notebook is a **feasibility/implementation demonstration**: it shows that every
component of the proposed architecture (FinBERT text encoder, Wav2Vec2 audio
encoder, ViT visual encoder, cross-modal fusion, FAISS retrieval, SHAP
explainability) runs correctly end-to-end on real data. Accuracy claims at this
scale are not made; large-scale statistical evaluation remains future work,
consistent with Chapter 7 of the thesis.

**How to use:**
1. Runtime -> Change runtime type -> T4 GPU
2. Upload `multimodal_pilot_data.zip` when prompted (Cell 2)
3. Run all cells in order.

In [ ]:
!pip -q install transformers faiss-cpu shap scikit-learn
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NOT AVAILABLE - enable GPU runtime!')

In [ ]:
# Cell 2 - upload and unzip the multimodal pilot data
from google.colab import files
up = files.upload()   # choose multimodal_pilot_data.zip
import zipfile
with zipfile.ZipFile('multimodal_pilot_data.zip', 'r') as z:
    z.extractall('.')
import os
print('Audio files:', os.listdir('audio'))
print('Frame dirs:', os.listdir('frames'))

In [ ]:
# Cell 3 - the 5 real events (text matches thesis political_events.csv)
events = [
    {'id': 'fed_20151216', 'date': '2015-12-16',
     'text': "The Federal Reserve raises interest rates for the first time in nearly a decade, "
             "signalling confidence in the US recovery while stressing a gradual path."},
    {'id': 'fed_20181219', 'date': '2018-12-19',
     'text': "The Fed raises rates a fourth time in 2018 and signals further tightening, "
             "disappointing markets hoping for a dovish pivot."},
    {'id': 'fed_20220316', 'date': '2022-03-16',
     'text': "The Fed raises rates for the first time since 2018, beginning what it signals "
             "will be a sustained tightening cycle against inflation."},
    {'id': 'fed_20240918', 'date': '2024-09-18',
     'text': "The Fed begins its easing cycle with an outsized 50 basis point rate cut, "
             "declaring greater confidence that inflation is moving to target."},
    {'id': 'fed_20241218', 'date': '2024-12-18',
     'text': "The Federal Reserve cuts rates by 25 basis points but signals a slower pace "
             "of cuts in 2025, adopting a more hawkish tone."},
]
print(f'{len(events)} real events loaded')

In [ ]:
# Cell 4 - Text encoder (FinBERT), same protocol as the Text+RAF pilot (Ch7)
from transformers import AutoTokenizer, AutoModel
import torch, numpy as np

tok_text = AutoTokenizer.from_pretrained('ProsusAI/finbert')
enc_text = AutoModel.from_pretrained('ProsusAI/finbert').cuda().eval()

text_embs = []
with torch.no_grad():
    for ev in events:
        batch = tok_text(ev['text'], truncation=True, max_length=128, return_tensors='pt').to('cuda')
        out = enc_text(**batch)
        text_embs.append(out.last_hidden_state[:,0,:].cpu().numpy()[0])  # CLS token
text_embs = np.stack(text_embs)
print('Text embeddings:', text_embs.shape)

In [ ]:
# Cell 5 - Audio encoder (Wav2Vec2), real 3-minute press-conference audio clips
!pip -q install soundfile librosa
from transformers import Wav2Vec2Processor, Wav2Vec2Model
import librosa

proc_audio = Wav2Vec2Processor.from_pretrained('facebook/wav2vec2-base-960h')
enc_audio = Wav2Vec2Model.from_pretrained('facebook/wav2vec2-base-960h').cuda().eval()

audio_embs = []
with torch.no_grad():
    for ev in events:
        wav, sr = librosa.load(f"audio/{ev['id']}.m4a", sr=16000, duration=60)  # first 60s
        inputs = proc_audio(wav, sampling_rate=16000, return_tensors='pt').to('cuda')
        out = enc_audio(**inputs)
        emb = out.last_hidden_state.mean(dim=1).cpu().numpy()[0]  # mean-pool over time
        audio_embs.append(emb)
audio_embs = np.stack(audio_embs)
print('Audio embeddings:', audio_embs.shape)

In [ ]:
# Cell 6 - Visual encoder (ViT), real key-frames extracted at 1 frame/10s
from transformers import ViTImageProcessor, ViTModel
from PIL import Image
import glob

proc_vis = ViTImageProcessor.from_pretrained('google/vit-base-patch16-224')
enc_vis = ViTModel.from_pretrained('google/vit-base-patch16-224').cuda().eval()

visual_embs = []
with torch.no_grad():
    for ev in events:
        frame_paths = sorted(glob.glob(f"frames/{ev['id']}/*.jpg"))
        frame_vecs = []
        for fp in frame_paths:
            img = Image.open(fp).convert('RGB')
            inputs = proc_vis(images=img, return_tensors='pt').to('cuda')
            out = enc_vis(**inputs)
            frame_vecs.append(out.last_hidden_state[:,0,:].cpu().numpy()[0])  # CLS token
        visual_embs.append(np.mean(frame_vecs, axis=0))  # mean-pool over key-frames
        print(f"  {ev['id']}: {len(frame_paths)} frames processed")
visual_embs = np.stack(visual_embs)
print('Visual embeddings:', visual_embs.shape)

In [ ]:
# Cell 7 - Cross-modal attention fusion (real implementation, Eq. in Section 5.5)
import torch.nn.functional as F

def cross_attn(query, key_value, d_k=768):
    # query, key_value: (N, 768) treated as single-token sequences per event
    Q = torch.tensor(query, dtype=torch.float32).cuda()
    K = torch.tensor(key_value, dtype=torch.float32).cuda()
    V = K.clone()
    scores = (Q @ K.T) / (d_k ** 0.5)
    weights = F.softmax(scores, dim=-1)
    return (weights @ V).cpu().numpy(), weights.cpu().numpy()

ta, w_ta = cross_attn(text_embs, audio_embs)
at, w_at = cross_attn(audio_embs, text_embs)
tv, w_tv = cross_attn(text_embs, visual_embs)
vt, w_vt = cross_attn(visual_embs, text_embs)
av, w_av = cross_attn(audio_embs, visual_embs)
va, w_va = cross_attn(visual_embs, audio_embs)

fused_raw = np.concatenate([ta, at, tv, vt, av, va], axis=1)  # (5, 6*768)
print('Fused (pre-projection) shape:', fused_raw.shape)

# Linear projection to 1024-dim (Section 5.5), random-init since no training data at N=5
torch.manual_seed(42)
proj = torch.nn.Linear(fused_raw.shape[1], 1024).cuda()
with torch.no_grad():
    fused = proj(torch.tensor(fused_raw, dtype=torch.float32).cuda()).cpu().numpy()
print('F_fused shape:', fused.shape)
print()
print('NOTE: with N=5 events there is no independent training set to learn the fusion')
print('projection weights; this cell demonstrates that the architecture executes correctly')
print('end-to-end on real data (dimensions match Section 5.5 exactly), not a trained model.')

In [ ]:
# Cell 8 - FAISS retrieval demonstration (real retrieval over the 5-event set + text-only 202-event KB)
import faiss

# Retrieve over the fused multimodal query against the fused representations themselves
# (leave-one-out: for each event, retrieve the most similar OTHER event)
Fn = fused / np.linalg.norm(fused, axis=1, keepdims=True)
index = faiss.IndexFlatIP(Fn.shape[1])
index.add(Fn.astype('float32'))

print('Leave-one-out nearest-neighbour retrieval among the 5 multimodal events:')
for i, ev in enumerate(events):
    sims, idxs = index.search(Fn[i:i+1].astype('float32'), 2)  # top-2 (self + nearest other)
    j = idxs[0][1] if idxs[0][0] == i else idxs[0][0]
    sim = sims[0][1] if idxs[0][0] == i else sims[0][0]
    print(f"  {ev['date']} ({ev['text'][:40]}...)")
    print(f"    -> nearest: {events[j]['date']} (sim={sim:.3f})")

In [ ]:
# Cell 9 - SHAP explainability on the fused multimodal representation
import shap

# Use a simple linear probe (financial direction is not fit at N=5; this demonstrates
# that SHAP runs correctly on the true 1024-dim fused multimodal representation)
from sklearn.linear_model import LogisticRegression
dummy_labels = np.array([1,0,1,0,1])  # placeholder direction labels for architecture demo only
clf = LogisticRegression(max_iter=1000).fit(fused, dummy_labels)

explainer = shap.LinearExplainer(clf, fused)
shap_vals = explainer.shap_values(fused)
print('SHAP values shape:', shap_vals.shape, '(N events x 1024 fused dims)')
print()
print('Mean |SHAP| per modality-pair block (indicates which cross-attention pairing')
print('the classifier would weight most heavily once trained on real labels):')
# fused was built by a random projection over 6 concatenated 768-dim blocks -> cannot
# cleanly attribute back to blocks post-projection; report aggregate magnitude instead.
print('Aggregate mean |SHAP| across all fused dims:', np.abs(shap_vals).mean())

In [ ]:
# Cell 10 - save results and download
import pandas as pd
summary = pd.DataFrame({
    'event_date': [e['date'] for e in events],
    'text_emb_dim': [text_embs.shape[1]]*5,
    'audio_emb_dim': [audio_embs.shape[1]]*5,
    'visual_emb_dim': [visual_embs.shape[1]]*5,
    'fused_dim': [fused.shape[1]]*5,
})
summary.to_csv('multimodal_pilot_summary.csv', index=False)
print(summary.to_string(index=False))

np.savez('multimodal_pilot_embeddings.npz',
        text=text_embs, audio=audio_embs, visual=visual_embs, fused=fused)

from google.colab import files
files.download('multimodal_pilot_summary.csv')
files.download('multimodal_pilot_embeddings.npz')